# eVoto — il protocollo a numeri piccoli

Questo notebook segue il protocollo di eVoto passo per passo, con numeri abbastanza piccoli da poter essere ricontrollati a mano. È l'esempio della sezione 38 della [specifica](../docs/spec_f1.md): tre garanti, ne bastano due per decifrare, tre elettori in un referendum sì/no.

Tutti i valori sono calcolati dalla libreria `evoto`. Le celle contengono dei controlli (`assert`) che li confrontano con quelli scritti nella specifica: se una formula cambiasse, il notebook se ne accorgerebbe.

Per aprirlo, dalla radice del repository:

```bash
uv run --with jupyter jupyter lab notebook/demo_didattica.ipynb
```

1. Il gruppo
2. Perché non `g = 2`
3. ElGamal esponenziale
4. Conteggio omomorfico
5. Shamir con un dealer
6. Cerimonia delle chiavi senza dealer
7. Decifratura a soglia con le prove
8. Una prova a conoscenza zero costruita a mano
9. Una scheda politica in piccolo
10. Bacheca e codici di tracciamento
11. Un'elezione completa e il verificatore indipendente

In [1]:
import sys
from pathlib import Path

# Il progetto non è installato come pacchetto: aggiungiamo agli import
# la radice del repository, cioè la prima cartella che contiene evoto/.
ROOT = next(
    folder
    for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "evoto").is_dir()
)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

## 1. Il gruppo

Lavoriamo in un sottogruppo di ordine primo `q` del gruppo moltiplicativo modulo `p`. Qui `p = 2579 = 2 · 1289 + 1`, lo stesso primo dell'esempio ElGamal della Lezione 7, e `q = 1289` è primo.

Il generatore è `g = 4 = 2²`: genera il sottogruppo dei quadrati, che ha ordine `q`. Tutti gli esponenti si riducono modulo `q`.

In un sistema reale `p` ha 2048 o 4096 bit (`DEMO_PARAMS` ne ha 2048, con `q` da 256 bit): la struttura è la stessa.

In [2]:
from evoto.gruppo import (
    TEST_PARAMS,
    is_subgroup_element,
    mod_inverse,
    mod_pow,
    validate_group_parameters,
)

params = TEST_PARAMS
p, q, g = params.p, params.q, params.g

print(f"p = {p}, q = {q}, g = {g}")
print("p = 2q + 1:", p == 2 * q + 1)
print("g^q mod p =", mod_pow(g, q, p))
print("parametri validi:", validate_group_parameters(params))

assert p == 2 * q + 1
assert mod_pow(g, q, p) == 1
assert validate_group_parameters(params)

p = 2579, q = 1289, g = 4
p = 2q + 1: True
g^q mod p = 1
parametri validi: True


Ogni valore ricevuto (cifrati, chiavi, impegni delle prove) viene controllato con `is_subgroup_element`, cioè `x^q = 1 mod p`. Il valore `2578 = -1` e il valore `2` non stanno nel sottogruppo: un avversario potrebbe usarli per far trapelare informazione.

In [3]:
for value in (4, 530, 2, 2578):
    print(f"{value:>4} nel sottogruppo: {is_subgroup_element(value, params)}")

assert is_subgroup_element(530, params)
assert not is_subgroup_element(2, params)
assert not is_subgroup_element(p - 1, params)

   4 nel sottogruppo: True
 530 nel sottogruppo: True
   2 nel sottogruppo: False
2578 nel sottogruppo: False


## 2. Perché non `g = 2`

Nella slide della Lezione 7 il generatore è `g = 2`, che genera tutto il gruppo modulo `p`, di ordine pari `2578`. Con un gruppo di ordine pari la cifratura fa trapelare un bit: il simbolo di Legendre `x^((p-1)/2) mod p` vale `1` se `x = 2^e` con `e` pari e `-1` se `e` è dispari.

Con `alpha = 2^r`, `K = 2^s` e `beta = 2^v · K^r = 2^(v + s·r)`, chiunque legge la parità di `r`, di `s` e di `v + s·r`, e quindi ricava la parità di `v`. In un voto sì/no la parità è il voto. È l'errore del sistema di voto elettronico di Mosca del 2019.

In [4]:
import random


def exponent_parity(value: int) -> int:
    """
    Parità dell'esponente e di value = 2^e mod p, letta dal simbolo
    di Legendre: 0 se e è pari, 1 se è dispari.
    """

    return 0 if pow(value, (p - 1) // 2, p) == 1 else 1


rng = random.Random(2019)

bad_g = 2
bad_secret = rng.randrange(1, p - 1)
bad_key = pow(bad_g, bad_secret, p)

secret_votes = [1, 0, 0, 1, 1, 0, 1, 0]
recovered = []

for vote in secret_votes:
    r = rng.randrange(1, p - 1)
    alpha = pow(bad_g, r, p)
    beta = pow(bad_g, vote, p) * pow(bad_key, r, p) % p

    # L'attaccante conosce solo alpha, beta e la chiave pubblica.
    recovered.append(
        exponent_parity(beta)
        ^ (exponent_parity(bad_key) & exponent_parity(alpha))
    )

print("voti segreti:", secret_votes)
print("ricostruiti :", recovered)

assert recovered == secret_votes

voti segreti: [1, 0, 0, 1, 1, 0, 1, 0]
ricostruiti : [1, 0, 0, 1, 1, 0, 1, 0]


Con `g = 4` ogni elemento del sottogruppo è un quadrato: il simbolo di Legendre vale sempre `1` e non dice nulla. Per questo serve un sottogruppo di ordine **primo**.

In [5]:
symbols = {
    pow(mod_pow(g, exponent, p), (p - 1) // 2, p)
    for exponent in range(q)
}

print("simboli di Legendre nel sottogruppo generato da 4:", symbols)

assert symbols == {1}

simboli di Legendre nel sottogruppo generato da 4: {1}


## 3. ElGamal esponenziale

Per ora la chiave segreta è un solo numero, `s = 765` (quello della slide), e la chiave pubblica è `K = g^s mod p`.

Un voto `v` si cifra con un numero casuale `R`, mettendo `v` all'esponente:

```text
Enc(v) = (alpha, beta) = (g^R mod p,  g^v · K^R mod p)
```

Cifriamo i voti `1, 0, 1` con `R = 11, 22, 33`.

In [6]:
from evoto.elgamal import (
    bounded_discrete_log,
    encrypt,
)

s = 765
K = mod_pow(g, s, p)

print("K = 4^765 mod 2579 =", K)
assert K == 530

votes = (1, 0, 1)
nonces = (11, 22, 33)

ciphertexts = tuple(
    encrypt(vote, K, params, nonce=nonce)
    for vote, nonce in zip(votes, nonces, strict=True)
)

for vote, nonce, ciphertext in zip(votes, nonces, ciphertexts, strict=True):
    print(f"Enc({vote}) con R = {nonce}: ({ciphertext.alpha}, {ciphertext.beta})")

assert [(c.alpha, c.beta) for c in ciphertexts] == [
    (850, 2375),
    (380, 22),
    (625, 670),
]

K = 4^765 mod 2579 = 530
Enc(1) con R = 11: (850, 2375)
Enc(0) con R = 22: (380, 22)
Enc(1) con R = 33: (625, 670)


Lo stesso calcolo a mano per il primo voto, e la decifratura: `beta / alpha^s = g^v`. Chi conosce `s` può decifrare ogni singola scheda: per questo nel sistema vero `s` non deve esistere in nessun posto (sezioni 5 e 6).

In [7]:
alpha = mod_pow(g, 11, p)
beta = mod_pow(g, 1, p) * mod_pow(K, 11, p) % p

print("a mano: (alpha, beta) =", (alpha, beta))
assert (alpha, beta) == (850, 2375)

g_to_v = beta * mod_inverse(mod_pow(alpha, s, p), p) % p
print("beta / alpha^s =", g_to_v, "= g^1")
assert g_to_v == g

a mano: (alpha, beta) = (850, 2375)
beta / alpha^s = 4 = g^1


## 4. Conteggio omomorfico

Moltiplicare due cifrati somma i voti:

```text
Enc(x) · Enc(y) = (g^(R1+R2),  g^(x+y) · K^(R1+R2)) = Enc(x + y)
```

Il prodotto dei tre cifrati è il totale cifrato `(A, B)`. Nessuna scheda viene decifrata: si decifra solo il totale, ricavando `t` da `g^t` con un logaritmo discreto **limitato**, facile perché `t` non supera il numero di schede (baby-step giant-step).

In [8]:
from evoto.urna import aggregate_ciphertexts

tally = aggregate_ciphertexts(ciphertexts, params)
A, B = tally.alpha, tally.beta

print("(A, B) =", (A, B))
assert (A, B) == (1196, 154)

g_to_t = B * mod_inverse(mod_pow(A, s, p), p) % p
t = bounded_discrete_log(g_to_t, params, max_exponent=len(votes))

print("B / A^s =", g_to_t, "= g^t con t =", t)
assert g_to_t == 16 and t == 2

(A, B) = (1196, 154)
B / A^s = 16 = g^t con t = 2


## 5. Shamir con un dealer

Dividiamo `s` tra tre garanti in modo che ne bastino due. Un **dealer** sceglie un polinomio di primo grado con `P(0) = s`

```text
P(x) = 765 + 100x mod 1289
```

e consegna al garante `l` la share `P(l)`. Due punti individuano una retta, uno solo no.

Per decifrare non si ricostruisce `s`: ogni garante presente pubblica `M_l = A^(P(l))` e si combinano le share all'esponente con i coefficienti di Lagrange valutati in zero, `λ_l = ∏ j / (j - l)` sui garanti presenti `j ≠ l`:

```text
M = ∏ M_l^(λ_l) = A^(Σ λ_l P(l)) = A^s        B / M = g^t
```

In [9]:
from evoto.decifratura import lagrange_coefficient

dealer_shares = {
    index: (765 + 100 * index) % q
    for index in (1, 2, 3)
}

print("share:", dealer_shares)
assert dealer_shares == {1: 865, 2: 965, 3: 1065}

present = (1, 3)

lambdas = {
    index: lagrange_coefficient(index, present, params)
    for index in present
}

print("garanti presenti:", present, " coefficienti:", lambdas)
assert lambdas == {1: 646, 3: 644}

# Solo per controllo: la combinazione delle share dà proprio s.
print("Σ λ_l P(l) mod q =", sum(lambdas[i] * dealer_shares[i] for i in present) % q)

partials = {
    index: mod_pow(A, dealer_shares[index], p)
    for index in present
}

print("share di decifratura M_l:", partials)
assert partials == {1: 60, 3: 1894}

M = 1
for index in present:
    M = M * mod_pow(partials[index], lambdas[index], p) % p

t = bounded_discrete_log(B * mod_inverse(M, p) % p, params, max_exponent=3)

print("M =", M, " t =", t, "con il garante 2 assente")
assert M == 332 and t == 2

share: {1: 865, 2: 965, 3: 1065}
garanti presenti: (1, 3)  coefficienti: {1: 646, 3: 644}
Σ λ_l P(l) mod q = 765
share di decifratura M_l: {1: 60, 3: 1894}
M = 332  t = 2 con il garante 2 assente


Qualunque coppia di garanti, o tutti e tre, ottiene lo stesso `M`: cambiano solo i coefficienti di Lagrange. È l'esperimento E2 in piccolo (`esperimenti/e2_garanti_assenti.py` lo ripete con 5 garanti e un'elezione vera).

In [10]:
print("presenti    λ_l               M")

for group in ((1, 2), (1, 3), (2, 3), (1, 2, 3)):
    coefficients = [lagrange_coefficient(i, group, params) for i in group]

    combined = 1
    for index, coefficient in zip(group, coefficients, strict=True):
        combined = combined * mod_pow(mod_pow(A, dealer_shares[index], p), coefficient, p) % p

    print(f"{str(group):<11} {str(coefficients):<17} {combined}")
    assert combined == 332

presenti    λ_l               M
(1, 2)      [2, 1288]         332
(1, 3)      [646, 644]        332
(2, 3)      [3, 1287]         332
(1, 2, 3)   [3, 1286, 1]      332


**Il problema del dealer.** Il dealer conosce `s`, quindi può decifrare ogni singola scheda: è la domanda «chi esegue il Setup?» della Lezione 11bis. La cerimonia delle chiavi elimina il dealer.

## 6. Cerimonia delle chiavi senza dealer

Ogni garante `i` sceglie da solo un polinomio `P_i(x) = a_i,0 + a_i,1 x`. Nell'esempio i polinomi sommano a quello del dealer, ma nessuno conosce la somma:

```text
P_1(x) = 300 + 40x      P_2(x) = 400 + 50x      P_3(x) = 65 + 10x
S(x) = P_1 + P_2 + P_3 = 765 + 100x
```

I passi sono:

1. ogni garante pubblica gli impegni `K_i,j = g^(a_i,j)`, con una prova di Schnorr per ciascuno;
2. manda al garante `l` la share `P_i(l)`, che `l` controlla con gli impegni (Feldman): `g^(P_i(l)) = ∏ K_i,j^(l^j)`;
3. ogni garante somma le share ricevute: `s_l = Σ_i P_i(l) = S(l)`;
4. la chiave pubblica è `K = ∏ K_i,0 = g^(S(0))`.

Le prove sono legate all'elezione dal contesto `Q = H(p, q, g, n, k, e)`, con `n = 3` garanti, quorum `k = 2` ed `e = 1`.

In [11]:
from evoto.garanti import (
    compute_base_hash,
    compute_commitments,
    create_guardian,
    create_guardian_record,
    verify_guardian_record,
)

COEFFICIENTS = ((300, 40), (400, 50), (65, 10))

guardians = [
    create_guardian(index=index, quorum=2, params=params, coefficients=coefficients)
    for index, coefficients in enumerate(COEFFICIENTS, start=1)
]

Q = compute_base_hash(params, guardian_count=3, quorum=2, election_id=1)
print("Q =", Q)
assert Q == 889

for guardian in guardians:
    print(f"garante {guardian.index}: K_i,0 = {compute_commitments(guardian, params)[0]:>4}, "
          f"K_i,1 = {compute_commitments(guardian, params)[1]:>4}")

assert [compute_commitments(guardian, params) for guardian in guardians] == [
    (2228, 1370),
    (523, 2277),
    (299, 1502),
]

Q = 889
garante 1: K_i,0 = 2228, K_i,1 = 1370
garante 2: K_i,0 =  523, K_i,1 = 2277
garante 3: K_i,0 =  299, K_i,1 = 1502


La prova di Schnorr dimostra di conoscere `a_i,j` senza rivelarlo. Per il garante 1 e l'impegno `K_1,0 = g^300`, con nonce `u = 7`:

```text
h = g^u = 910        c = H(Q, 1, 0, g, K_1,0, h) = 957        z = u + c · 300 mod q = 949
```

Chiunque controlla `g^z = h · K_1,0^c`.

In [12]:
records = [
    create_guardian_record(guardian, params, Q, nonces=(7, 7))
    for guardian in guardians
]

proof = records[0].proofs[0]
print("Schnorr del garante 1 su K_1,0:", (proof.commitment, proof.challenge, proof.response))
assert (proof.commitment, proof.challenge, proof.response) == (910, 957, 949)

left = mod_pow(g, proof.response, p)
right = proof.commitment * mod_pow(2228, proof.challenge, p) % p
print("g^z =", left, " h · K^c =", right)
assert left == right

print("impegni e prove validi:", [verify_guardian_record(r, 2, params, Q) for r in records])
assert all(verify_guardian_record(record, 2, params, Q) for record in records)

Schnorr del garante 1 su K_1,0: (910, 957, 949)
g^z = 1761  h · K^c = 1761
impegni e prove validi: [True, True, True]


Le share private e il controllo di Feldman. Una share alterata viene scoperta subito, e si sa chi l'ha mandata.

In [13]:
from evoto.garanti import (
    aggregate_shares,
    compute_share,
    verify_share,
)

shares = {
    (guardian.index, recipient): compute_share(guardian, recipient, params)
    for guardian in guardians
    for recipient in (1, 2, 3)
}

print("          a 1   a 2   a 3")
for guardian in guardians:
    row = "  ".join(f"{shares[guardian.index, recipient]:>4}" for recipient in (1, 2, 3))
    print(f"da {guardian.index}:   {row}")

assert [shares[1, recipient] for recipient in (1, 2, 3)] == [340, 380, 420]
assert [shares[2, recipient] for recipient in (1, 2, 3)] == [450, 500, 550]
assert [shares[3, recipient] for recipient in (1, 2, 3)] == [75, 85, 95]

# Feldman a mano: g^(P_1(2)) = K_1,0 · K_1,1^2
print("g^380 =", mod_pow(g, 380, p), " K_1,0 · K_1,1^2 =", 2228 * mod_pow(1370, 2, p) % p)

ok = all(
    verify_share(shares[sender, recipient], recipient, records[sender - 1], params)
    for sender in (1, 2, 3)
    for recipient in (1, 2, 3)
)
tampered = verify_share(shares[1, 2] + 1, 2, records[0], params)

print("tutte le share valide:", ok, " share alterata accettata:", tampered)
assert ok and not tampered

          a 1   a 2   a 3
da 1:    340   380   420
da 2:    450   500   550
da 3:     75    85    95
g^380 = 755  K_1,0 · K_1,1^2 = 755
tutte le share valide: True  share alterata accettata: False


Le share aggregate coincidono con quelle del dealer, ma questa volta nessuno ha mai visto `s`. Le chiavi di verifica `V_l = g^(s_l)` si calcolano dai soli impegni pubblici, e servono a controllare le share di decifratura.

In [14]:
from evoto.garanti import (
    compute_extended_base_hash,
    compute_joint_public_key,
    compute_verification_key,
    run_key_ceremony,
)

aggregated = {
    index: aggregate_shares(tuple(shares[sender, index] for sender in (1, 2, 3)), params)
    for index in (1, 2, 3)
}
verification_keys = {
    index: compute_verification_key(index, tuple(records), params)
    for index in (1, 2, 3)
}

K_joint = compute_joint_public_key(tuple(records), params)
Q_bar = compute_extended_base_hash(Q, K_joint, params)

print("share aggregate s_l:", aggregated)
print("chiavi di verifica V_l:", verification_keys)
print("K =", K_joint, " Q_bar = H(Q, K) =", Q_bar)

assert aggregated == {1: 865, 2: 965, 3: 1065}
assert verification_keys == {1: 2502, 2: 2488, 3: 2237}
assert K_joint == 530 and Q_bar == 744

# La libreria esegue tutti i passi insieme con run_key_ceremony.
ceremony = run_key_ceremony(3, 2, params, election_id=1, coefficients=COEFFICIENTS)
assert ceremony.joint_public_key == 530
assert ceremony.extended_base_hash == 744
assert ceremony.secret_shares == aggregated

share aggregate s_l: {1: 865, 2: 965, 3: 1065}
chiavi di verifica V_l: {1: 2502, 2: 2488, 3: 2237}
K = 530  Q_bar = H(Q, K) = 744


## 7. Decifratura a soglia con le prove

Ogni garante presente pubblica `M_l = A^(s_l)` con una prova di **Chaum-Pedersen**: lo stesso esponente `s_l` lega `V_l = g^(s_l)` e `M_l = A^(s_l)`. Così nessun garante può pubblicare un valore sbagliato. Per il garante 1, con nonce `u = 7`:

```text
a = g^u = 910     b = A^u = 1033     c = H(Q_bar, 1, g, V_1, A, M_1, a, b) = 524     z = u + c · s_1 mod q = 828
```

Il garante 2 è assente.

In [15]:
from evoto.decifratura import (
    compute_decryption_share,
    decrypt_tally,
    verify_decryption_share,
)

share_1 = compute_decryption_share(1, 865, tally, params, Q_bar, nonce=7)
share_3 = compute_decryption_share(3, 1065, tally, params, Q_bar, nonce=11)

cp = share_1.proof
print("M_1 =", share_1.partial_decryption,
      " prova (a, b, c, z) =", (cp.commitment_1, cp.commitment_2, cp.challenge, cp.response))
assert share_1.partial_decryption == 60
assert (cp.commitment_1, cp.commitment_2, cp.challenge, cp.response) == (910, 1033, 524, 828)

valid = [
    verify_decryption_share(share, verification_keys[share.guardian_index], tally, params, Q_bar)
    for share in (share_1, share_3)
]
print("prove delle share valide:", valid)
assert all(valid)

t = decrypt_tally(tally, (share_1, share_3), quorum=2, params=params, max_total=3)
print("totale decifrato con i garanti 1 e 3:", t)
assert t == 2

M_1 = 60  prova (a, b, c, z) = (910, 1033, 524, 828)
prove delle share valide: [True, True]
totale decifrato con i garanti 1 e 3: 2


Un garante che pubblica un valore diverso da `A^(s_l)` non ha una prova valida, e un solo garante non basta per decifrare.

In [16]:
from dataclasses import replace

cheating = replace(share_3, partial_decryption=share_3.partial_decryption * g % p)
print("share alterata accettata:",
      verify_decryption_share(cheating, verification_keys[3], tally, params, Q_bar))
assert not verify_decryption_share(cheating, verification_keys[3], tally, params, Q_bar)

try:
    decrypt_tally(tally, (share_1,), quorum=2, params=params, max_total=3)
except ValueError as error:
    print("con un solo garante:", error)

share alterata accettata: False
con un solo garante: I garanti presenti sono meno del quorum.


Con una sola share, ogni possibile segreto resta ugualmente probabile: per ognuno dei 1289 valori di `s` c'è esattamente una retta che passa per `(1, 865)`. È la seconda parte dell'esperimento E2. Con gli impegni di Feldman, però, `K = g^s` è pubblica: il segreto è protetto dal logaritmo discreto, non dall'informazione.

In [17]:
from esperimenti.e2_garanti_assenti import count_compatible_polynomials

one_share = count_compatible_polynomials({1: 865}, quorum=2, q=q)
two_shares = count_compatible_polynomials({1: 865, 3: 1065}, quorum=2, q=q)

print("con la share del garante 1:", len(one_share), "segreti possibili, ognuno con",
      one_share[765], "polinomio compatibile")
print("con le share dei garanti 1 e 3:", dict(two_shares))

assert len(one_share) == q and set(one_share.values()) == {1}
assert two_shares == {765: 1}

con la share del garante 1: 1289 segreti possibili, ognuno con 1 polinomio compatibile
con le share dei garanti 1 e 3: {765: 1}


## 8. Una prova a conoscenza zero costruita a mano

Ogni casella della scheda deve contenere `0` oppure `1`, ma nessuno deve sapere quale. La prova OR (Cramer, Damgård e Schoenmakers, 1994) ha un ramo per ogni valore ammesso:

- il ramo **vero** si dimostra onestamente, con un nonce `u`;
- gli altri rami si **simulano**: si scelgono prima sfida e risposta, poi si calcolano gli impegni che rendono vere le equazioni;
- la sfida di Fiat-Shamir `c = H(Q_bar, alpha, beta, a_0, b_0, a_1, b_1)` deve essere la somma delle sfide dei rami, quindi chi dimostra può scegliere liberamente tutte le sfide tranne una: quella del ramo vero.

Costruiamo a mano la prova per il primo voto, `Enc(1)` con `R = 11`, scegliendo `u = 7` e, per il ramo simulato `v = 0`, sfida `100` e risposta `200`. Poi la facciamo controllare dalla libreria.

In [18]:
from evoto.gruppo import H
from evoto.prove import (
    ValueSetBranchProof,
    ValueSetProof,
    verify_value_in_set,
)

vote_ciphertext = ciphertexts[0]          # Enc(1) con R = 11
alpha, beta = vote_ciphertext.alpha, vote_ciphertext.beta

# Ramo simulato v = 0: sfida e risposta scelte prima degli impegni.
c_0, z_0 = 100, 200
a_0 = mod_pow(g, z_0, p) * mod_inverse(mod_pow(alpha, c_0, p), p) % p
b_0 = mod_pow(K, z_0, p) * mod_inverse(mod_pow(beta, c_0, p), p) % p

# Ramo vero v = 1: impegni con il nonce u.
u = 7
a_1 = mod_pow(g, u, p)
b_1 = mod_pow(K, u, p)

c = H(Q_bar, alpha, beta, a_0, b_0, a_1, b_1, params=params)
c_1 = (c - c_0) % q
z_1 = (u + c_1 * 11) % q

print("sfida di Fiat-Shamir c =", c, "=", c_0, "+", c_1, "mod q")

proof = ValueSetProof(
    branches=(
        ValueSetBranchProof(commitment_1=a_0, commitment_2=b_0, challenge=c_0, response=z_0),
        ValueSetBranchProof(commitment_1=a_1, commitment_2=b_1, challenge=c_1, response=z_1),
    )
)

accepted = verify_value_in_set(vote_ciphertext, proof, (0, 1), K, params, Q_bar)
print("la libreria accetta la prova:", accepted)
assert accepted

sfida di Fiat-Shamir c = 1181 = 100 + 1081 mod q
la libreria accetta la prova: True


Dai numeri della prova non si capisce quale ramo sia quello vero: entrambi soddisfano le stesse equazioni. Ma senza conoscere `R` non si riesce a costruire una prova per un cifrato che contiene `2`, perché nessuno dei due rami può essere quello vero. E la stessa prova non vale per un altro cifrato, perché la sfida dipende da `alpha` e `beta` (Fiat-Shamir **forte**).

In [19]:
from evoto.prove import prove_value_in_set

two = encrypt(2, K, params, nonce=11)

try:
    prove_value_in_set(two, 2, 11, (0, 1), K, params, Q_bar)
except ValueError as error:
    print("Enc(2):", error)

print("la prova di Enc(1) vale per Enc(2)?", verify_value_in_set(two, proof, (0, 1), K, params, Q_bar))
assert not verify_value_in_set(two, proof, (0, 1), K, params, Q_bar)

Enc(2): Il plaintext non appartiene ai valori ammessi.
la prova di Enc(1) vale per Enc(2)? False


## 9. Una scheda politica in piccolo

Due liste, A e B, ciascuna con il capolista bloccato (che non ha casella) e due candidati, uno per genere. Al massimo 3 preferenze e al massimo 2 dello stesso genere. La scheda ha 7 caselle, quindi 7 cifrati:

```text
lista A, lista B, bianca, A1 (M), A2 (F), B1 (M), B2 (F)
```

Le regole diventano prove sui cifrati e sulle loro combinazioni omomorfiche:

| Regola | Vincolo | Prove |
|---|---|---|
| R1 | ogni casella in `{0, 1}` | 7 |
| R2 | `lista A + lista B + bianca = 1` | 1 |
| R3 | `lista P - preferenza c ∈ {0, 1}` per ogni candidato `c` della lista `P` | 4 |
| R4 | somma delle preferenze in `{0, ..., 3}` | 1 |
| R5 | somma delle preferenze di ogni genere in `{0, 1, 2}` | 2 |

Votiamo la lista B con una preferenza a B1.

In [20]:
from evoto.scheda import (
    BallotLayout,
    PreferenceMetadata,
    verify_ballot,
)
from evoto.voto import (
    VoterChoice,
    choice_to_plaintexts,
    prepare_ballot,
)

layout = BallotLayout(
    list_count=2,
    preference_metadata=(
        PreferenceMetadata(list_index=0, gender="M"),
        PreferenceMetadata(list_index=0, gender="F"),
        PreferenceMetadata(list_index=1, gender="M"),
        PreferenceMetadata(list_index=1, gender="F"),
    ),
    max_preferences=3,
    max_preferences_per_gender=2,
)

choice = VoterChoice(list_index=1, preferences=(2,))
print("bit (liste, bianca, preferenze):", choice_to_plaintexts(layout, choice))

ballot_nonces = (101, 102, 103, 104, 105, 106, 107)
prepared = prepare_ballot(layout, 0, choice, K_joint, params, Q_bar, nonces=ballot_nonces)

proofs = prepared.proofs
counts = (len(proofs.r1_proofs), 1, len(proofs.r3_proofs), 1, len(proofs.r5_proofs))
print("prove R1-R5:", counts, "totale", sum(counts))
print("scheda valida:", verify_ballot(layout, prepared.ballot, proofs, K_joint, params, Q_bar))

assert counts == (7, 1, 4, 1, 2)
assert verify_ballot(layout, prepared.ballot, proofs, K_joint, params, Q_bar)

bit (liste, bianca, preferenze): ((0, 1), 0, (0, 0, 1, 0))
prove R1-R5: (7, 1, 4, 1, 2) totale 15
scheda valida: True


Un dispositivo scorretto che prova a dare una preferenza ad A1 votando la lista B non riesce a costruire la prova R3: il cifrato derivato `lista A - A1` contiene `-1`, che non sta in `{0, 1}`. La validità della scheda non dipende dall'interfaccia (esperimento E4).

In [21]:
wrong = VoterChoice(list_index=1, preferences=(0,))

try:
    prepare_ballot(layout, 0, wrong, K_joint, params, Q_bar)
except ValueError as error:
    print("preferenza fuori lista:", error)

# Una scelta corretta con due preferenze, una per genere, invece è valida.
right = prepare_ballot(layout, 0, VoterChoice(list_index=0, preferences=(0, 1)), K_joint, params, Q_bar)
right_is_valid = verify_ballot(layout, right.ballot, right.proofs, K_joint, params, Q_bar)

print("lista A con preferenze ad A1 e A2: valida =", right_is_valid)
assert right_is_valid

preferenza fuori lista: Il plaintext non appartiene ai valori ammessi.
lista A con preferenze ad A1 e A2: valida = True


**Perché il conteggio omomorfico.** Con tre preferenze su otto candidati una lista offre `1 + 8 + 28 + 56 = 93` combinazioni: chi compra voti può assegnare a ogni elettore una combinazione e cercarla nello spoglio (*Italian attack*). L'attacco funziona solo se le singole schede vengono decifrate. Qui si decifrano solo i totali, e le combinazioni spariscono.

In [22]:
from math import comb

combinations = sum(comb(8, k) for k in range(4))
print("combinazioni di preferenze per lista:", combinations)
assert combinations == 93

combinazioni di preferenze per lista: 93


## 10. Bacheca e codici di tracciamento

Ogni scheda depositata riceve un codice concatenato al precedente, come in una catena di hash:

```text
code_0 = H(Q_bar)        code_i = H(code_(i-1), i, stato, h_i)
```

dove `h_i` è l'impronta della scheda e lo stato vale 1 per una scheda depositata (CAST) e 2 per una sprecata (SPOILED). L'elettore conserva il proprio codice e lo ritrova sulla bacheca; cambiare una riga qualsiasi rompe tutti i codici successivi.

Prima di depositare, l'elettore può **sprecare** la scheda (sfida di Benaloh): il dispositivo rivela i nonce e chiunque ricifra i voti per controllare che abbia cifrato onestamente. Una scheda sprecata non viene contata.

In [23]:
from evoto.urna import (
    cast_ballot,
    create_bulletin_board,
    spoil_ballot,
    verify_board_chain,
    verify_spoiled_ballot,
)

board = create_bulletin_board(Q_bar, params)
print("code_0 = H(Q_bar) =", board.genesis_code)

# Il primo elettore controlla il dispositivo: spreca una scheda, poi vota.
challenged = prepare_ballot(layout, 0, choice, K_joint, params, Q_bar, nonces=(11, 12, 13, 14, 15, 16, 17))
board = spoil_ballot(board, 0, layout, challenged.ballot, challenged.proofs, challenged.witness, K_joint, params)
print("scheda sprecata ricifrata correttamente:",
      verify_spoiled_ballot(challenged.ballot, challenged.witness, K_joint, params))

board = cast_ballot(board, 0, layout, prepared.ballot, prepared.proofs, K_joint, params)

second = prepare_ballot(layout, 0, VoterChoice(list_index=None), K_joint, params, Q_bar,
                        nonces=(21, 22, 23, 24, 25, 26, 27))
board = cast_ballot(board, 0, layout, second.ballot, second.proofs, K_joint, params)

for entry in board.entries:
    print(f"{entry.sequence}  {entry.state:<8} impronta {entry.ballot_hash:>4}  codice {entry.tracking_code:>4}")

print("catena valida:", verify_board_chain(board, params))
assert verify_board_chain(board, params)

code_0 = H(Q_bar) = 631


scheda sprecata ricifrata correttamente: True
1  SPOILED  impronta  617  codice  581
2  CAST     impronta  730  codice   32
3  CAST     impronta  328  codice  353
catena valida: True


In [24]:
from dataclasses import replace

entries = list(board.entries)
entries[1] = replace(entries[1], tracking_code=(entries[1].tracking_code + 1) % q)
forged = replace(board, entries=tuple(entries))

print("catena valida dopo la modifica di una riga:", verify_board_chain(forged, params))
assert not verify_board_chain(forged, params)

try:
    cast_ballot(board, 0, layout, prepared.ballot, prepared.proofs, K_joint, params)
except ValueError as error:
    print("scheda ricopiata:", error)

catena valida dopo la modifica di una riga: False
scheda ricopiata: La scheda è già presente sulla bacheca.


## 11. Un'elezione completa e il verificatore indipendente

Mettiamo insieme tutto con la configurazione di esempio (tre circoscrizioni, otto liste, due coalizioni): pochi elettori simulati, cinque garanti, ne bastano tre. Il registro pubblico in JSON contiene solo dati pubblici; il verificatore in `verifica/verifica.py` lo ricontrolla da solo, senza importare `evoto`.

In [25]:
import json

from evoto.configurazione import load_election_config
from evoto.registro import public_registry_to_json
from evoto.simulazione import simulate_election
from verifica.verifica import verify_public_registry

config = load_election_config(ROOT / "config" / "elezione_esempio.json")

report = simulate_election(
    config=config,
    voters_per_district=10,
    guardian_count=5,
    quorum=3,
    present_guardians=(1, 3, 5),
    params=params,
    seed=2026,
)

print("schede depositate:", sum(result.ballot_count for result in report.results))
print("seggi per lista:", dict(zip(config.list_names, report.scrutiny.list_seats, strict=True)))

registry = public_registry_to_json(report, params)
verdict = verify_public_registry(registry)

print(verdict)
print("ELEZIONE VERIFICATA" if verdict["overall"] else "VERIFICA FALLITA")
assert verdict["overall"]

schede depositate: 30
seggi per lista: {'Lista A': 4, 'Lista B': 2, 'Lista C': 2, 'Lista D': 3, 'Lista E': 9, 'Lista F': 4, 'Lista G': 4, 'Lista H': 2}


{'V1': True, 'V2': True, 'V3': True, 'V4': True, 'V5': True, 'V6': True, 'V7': True, 'V8': True, 'overall': True}
ELEZIONE VERIFICATA


Infine alteriamo il registro: un voto in più alla prima lista della prima circoscrizione. Il totale non è più compatibile con la decifratura (V7) e il risultato dello scrutinio cambia (V8).

In [26]:
data = json.loads(registry)
data["district_results"][0]["list_votes"][0] += 1

tampered_verdict = verify_public_registry(json.dumps(data))

print(tampered_verdict)
print("ELEZIONE VERIFICATA" if tampered_verdict["overall"] else "VERIFICA FALLITA")
assert not tampered_verdict["V7"] and not tampered_verdict["V8"]

{'V1': True, 'V2': True, 'V3': True, 'V4': True, 'V5': True, 'V6': True, 'V7': False, 'V8': False, 'overall': False}
VERIFICA FALLITA


## Riepilogo

| Fase | Cosa abbiamo visto | Lezione |
|---|---|---|
| Gruppo | sottogruppo di ordine primo; con ordine pari un bit trapela (Mosca 2019) | 7, 11bis |
| Cerimonia | Shamir senza dealer, impegni di Feldman, prove di Schnorr | 14, 11bis, 12 |
| Voto | ElGamal esponenziale, prove OR rese non interattive con Fiat-Shamir | 7, 12 |
| Bacheca | catena di hash, sfida di Benaloh | 6, 11 |
| Conteggio | prodotto dei cifrati = cifrato della somma | 13 |
| Decifratura | Lagrange all'esponente, prove di Chaum-Pedersen, garanti assenti | 14, 12 |
| Verifica | tutto si ricontrolla dal registro pubblico | — |

Le formule complete sono nella [specifica](../docs/spec_f1.md); gli esperimenti E2 ed E5 sono in `esperimenti/`.